In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/leash-BELKA/sample_submission.csv
/kaggle/input/leash-BELKA/train.parquet
/kaggle/input/leash-BELKA/test.parquet
/kaggle/input/leash-BELKA/train.csv
/kaggle/input/leash-BELKA/test.csv
/kaggle/input/models/t8101349/prime-model/pytorch/default/1/xgb_fold_4.json
/kaggle/input/models/t8101349/prime-model/pytorch/default/1/cnn_fold_3.pt
/kaggle/input/models/t8101349/prime-model/pytorch/default/1/xgb_fold_2.json
/kaggle/input/models/t8101349/prime-model/pytorch/default/1/cnn_fold_1.pt
/kaggle/input/models/t8101349/prime-model/pytorch/default/1/lgbm_fold_5.txt
/kaggle/input/models/t8101349/prime-model/pytorch/default/1/xgb_fold_3.json
/kaggle/input/models/t8101349/prime-model/pytorch/default/1/lgbm_fold_4.txt
/kaggle/input/models/t8101349/prime-model/pytorch/default/1/lgbm_fold_1.txt
/kaggle/input/models/t8101349/prime-model/pytorch/default/1/lgbm_fold_2.txt
/kaggle/input/models/t8101349/prime-model/pytorch/default/1/xgb_fold_1.json
/kaggle/input/models/t8101349/prime-model/pyto

In [2]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 45.4 MB/s eta 0:00:00:00:0100:01


In [3]:
import os
import time
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from tqdm import tqdm
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem import Descriptors
import torch.nn.functional as F
from sklearn.model_selection import GroupKFold
from sklearn.metrics import average_precision_score 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
import gc

In [4]:
# --- 1. 數據優化抽樣 (Optimized Sampling) ---
# 參數設定
INPUT_FILE = "/kaggle/input/leash-BELKA/train.csv"
OUTPUT_FILE = "optimized_train_data.csv"
CHUNK_SIZE = 1_000_000
NEG_TO_POS_RATIO = 4  # 負正比設定為 20:1 (配合 Focal Loss)

def optimized_sampling():
    # 1. 快速過濾所有正樣本 
    print("🚀 提取所有正樣本...")
    pos_chunks = []
    for chunk in tqdm(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE), desc="Extracting Positives"):
        pos_chunks.append(chunk[chunk['binds'] == 1])
    
    df_pos = pd.concat(pos_chunks)
    total_pos_count = len(df_pos)
    print(f"✅ 找到 {total_pos_count} 個正樣本")

    # 2. 針對負樣本進行隨機/分層採樣
    # 為了效能，我們在讀取時直接進行伯努利採樣 (Bernoulli Sampling)
    target_neg_count = total_pos_count * NEG_TO_POS_RATIO
    # 計算抽樣機率 (總數據約 1.3 億*3)
    sampling_rate = target_neg_count / 390_000_000 
    
    print(f"🎲 正在以 {sampling_rate:.4f} 的機率採樣負樣本...")
    neg_chunks = []
    for chunk in tqdm(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE), desc="Sampling Negatives"):
        neg_subset = chunk[chunk['binds'] == 0]
        # 使用隨機掩碼進行快速採樣，比 groupby 快得多
        sampled_neg = neg_subset.sample(frac=sampling_rate, random_state=42)
        neg_chunks.append(sampled_neg)
        
    df_neg = pd.concat(neg_chunks)
    
    # 3. 合併並洗牌
    df_final = pd.concat([df_pos, df_neg]).sample(frac=1, random_state=42).reset_index(drop=True)
    
    # 4. 蛋白質特徵工程 (順便完成，減少之後讀取壓力)
    protein_map = {'BRD4': 0, 'HSA': 1, 'sEH': 2}
    df_final['protein_id'] = df_final['protein_name'].map(protein_map).astype(np.int8)
    df_final['binds'] = df_final['binds'].astype(np.int8)
    
    print(f"📊 最終樣本數: {len(df_final)} (正: {len(df_pos)}, 負: {len(df_neg)})")
    df_final.to_csv(OUTPUT_FILE, index=False)
    print(f"✅ 檔案已儲存至 {OUTPUT_FILE}")

# 執行
optimized_sampling()

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

🚀 提取所有正樣本...


Extracting Positives: 296it [19:16,  3.91s/it]


✅ 找到 1589906 個正樣本
🎲 正在以 0.0163 的機率採樣負樣本...


Sampling Negatives: 296it [16:03,  3.26s/it]


📊 最終樣本數: 6378492 (正: 1589906, 負: 4788586)
✅ 檔案已儲存至 optimized_train_data.csv


In [5]:

# --- 2. 蛋白質物理特徵工程 (Protein Engineering) ---
protein_info = {
    'BRD4': {'MW_kDa': 15.5, 'pI': 9.67, 'gravy': -0.82}, 
    'HSA':  {'MW_kDa': 66.4, 'pI': 4.70, 'gravy': -0.44}, 
    'sEH':  {'MW_kDa': 62.6, 'pI': 5.89, 'gravy': -0.22}
}

def extract_protein_features(df):
    # 修正：統一使用 MW_kDa
    for prop in ['MW_kDa', 'pI', 'gravy']:
        df[f'prot_{prop}'] = df['protein_name'].apply(lambda x: protein_info[x][prop])
    
    # 蛋白質 ID (給 CNN Embedding 層用)
    protein_map = {'BRD4': 0, 'HSA': 1, 'sEH': 2}
    df['protein_id'] = df['protein_name'].map(protein_map).astype(np.int8)
    
    # 數值特徵標準化
    scaler = StandardScaler()
    num_cols = ['prot_MW_kDa', 'prot_pI', 'prot_gravy']
    df[num_cols] = scaler.fit_transform(df[num_cols])
    return df, num_cols

In [6]:
# --- 3. 摩根指紋生成 (Fingerprints for GBDT) ---
fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def smiles_to_fp_fast(smiles_list):
    fps = []
    for s in smiles_list:
        mol = Chem.MolFromSmiles(s)
        if mol:
            # 使用 GetCountFingerprintAsNumPy 獲取計數指紋，或者 GetFingerprintAsNumPy 獲取位元指紋
            fps.append(np.array(fp_gen.GetFingerprintAsNumPy(mol), dtype=np.int8))
        else:
            fps.append(np.zeros(2048, dtype=np.int8))
    return np.vstack(fps)


In [7]:
# --- 4. 1D-CNN 融合模型 (Fusion Model) ---
class BelkaFusionModel(nn.Module):
    def __init__(self, vocab_size, num_prot_features=3, embed_dim=128):
        super(BelkaFusionModel, self).__init__()
        self.smiles_embed = nn.Embedding(vocab_size, embed_dim)
        self.cnn = nn.Sequential(
            nn.Conv1d(embed_dim, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), 
            nn.GELU(),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128), 
            nn.GELU(),
            nn.AdaptiveMaxPool1d(1)
        )
        self.protein_id_embed = nn.Embedding(3, 16) 
        self.protein_numeric_mlp = nn.Sequential(
            nn.Linear(num_prot_features, 16),
            nn.BatchNorm1d(16),  
            nn.GELU()
        )
        
        self.fc = nn.Sequential(
            nn.Linear(128 + 16 + 16, 64),
            nn.BatchNorm1d(64),  
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(64, 1)
        )
        
    def forward(self, smiles_seq, protein_id, prot_numeric):
        x = self.smiles_embed(smiles_seq).permute(0, 2, 1) 
        x = self.cnn(x).squeeze(-1)
        p_id = self.protein_id_embed(protein_id)
        p_num = self.protein_numeric_mlp(prot_numeric)
        combined = torch.cat([x, p_id, p_num], dim=1)
        return self.fc(combined) # 用於 Focal Loss 時不加 Sigmoid

In [ ]:

from scipy import sparse
import gc

input_file = "optimized_train_data.csv"
df_sampled = pd.read_csv(input_file)
df, prot_num_cols = extract_protein_features(df_sampled)
num_samples = len(df_sampled)

# 1. 建立儲存清單
sparse_list = []
batch_size = 50000

print(f"🚀 開始以稀疏格式處理 {len(df_sampled)} 筆數據...")

for i in tqdm(range(0, len(df_sampled), batch_size)):
    end_i = min(i + batch_size, len(df_sampled))
    batch_smiles = df_sampled['molecule_smiles'].iloc[i:end_i].values
    
    # 生成指紋 (預設是密集矩陣)
    batch_fps = smiles_to_fp_fast(batch_smiles)
    
    # 轉換為 CSR 稀疏格式
    # 這會瞬間釋放掉 0 所佔用的空間
    batch_fps_sparse = sparse.csr_matrix(batch_fps.astype('uint8'))
    sparse_list.append(batch_fps_sparse)

# 2. 合併所有分塊
print("📦 正在合併分塊...")
X_fp_sparse = sparse.vstack(sparse_list)

del sparse_list
gc.collect()

# 3. 儲存成 .npz 格式 (這比 .npy 小得多！)
sparse.save_npz('X_fp_sparse.npz', X_fp_sparse)

del X_fp_sparse
gc.collect()

df[prot_num_cols].to_parquet('X_prot_features.parquet')
df[['binds', 'buildingblock1_smiles']].to_parquet('y_train.parquet')


del df
gc.collect()
print("✅ 稀疏矩陣儲存完成！可以開始訓練了。")

In [ ]:
# A. 載入特徵
X_fp = sparse.load_npz('X_fp_sparse.npz')
X_fp = X_fp.astype(np.float32)
X_prot = pd.read_parquet('X_prot_features.parquet').values
X_final_sparse = sparse.hstack([X_fp, sparse.csr_matrix(X_prot)])

# B. 重新從原始 CSV 讀取標籤與 Group 資訊 (確保長度 100% 對齊)
# 不要從 y_train.parquet 讀，因為那個檔案顯然重複了
df_base = pd.read_csv("optimized_train_data.csv", usecols=['binds', 'buildingblock1_smiles'])

# C. 強制檢查與截斷
# 如果 CSV 比 X 短或長，以 X 的數量為準
n_samples = X_final_sparse.shape[0]

y = df_base['binds'].values[:n_samples]
groups = df_base['buildingblock1_smiles'].values[:n_samples]

print(f"📊 修正後狀態:")
print(f"X_final_sparse: {X_final_sparse.shape}")
print(f"y: {y.shape}")
print(f"groups: {len(groups)}")

if X_final_sparse.shape[0] == y.shape[0]:
    print("✅ 長度已完美對齊，可以開始訓練！")
else:
    print("❌ 長度依然不符，請檢查 optimized_train_data.csv 的行數是否與 X_fp 一致。")

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import sparse
import gc
from sklearn.model_selection import GroupKFold
from sklearn.metrics import average_precision_score

gkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42) # n folds

lgb_params = {
    'objective': 'binary',
    'metric': 'auc',      # 監控指標
    'boosting_type': 'gbdt',
    'n_jobs': -1,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'min_child_samples': 20,
    'scale_pos_weight': 4, # 處理 20:1 的負正比
    'device': 'gpu',        # 如果有 GPU 請開啟gpu
    'max_bin': 63,
    'random_state': 42
}

# 2. 準備 OOF 容器
oof_lgb = np.zeros(len(y))

# 3. 5-Fold 訓練
for fold, (train_idx, val_idx) in enumerate(gkf.split(X_final_sparse, y, groups=groups)):
    print(f"\n🚀 --- Training LGBM Fold {fold+1} (Sparse Mode) ---")
    
    # --- 訓練集處理 ---
    # 分段合併並立即轉化為 Dataset
    X_train = sparse.hstack([X_fp[train_idx], sparse.csr_matrix(X_prot[train_idx])])
    y_train = y[train_idx]
    
    # 關鍵參數：free_raw_data=True 會在建立 Dataset 後釋放 X_train
    dtrain = lgb.Dataset(X_train, label=y_train, free_raw_data=True)
    
    # 建立 dtrain 後立即手動刪除 X_train，不要等迴圈結束
    del X_train, y_train
    gc.collect() 
    
    # --- 驗證集處理 ---
    X_val_raw = X_fp[val_idx]
    X_val_prot = sparse.csr_matrix(X_prot[val_idx])
    X_val = sparse.hstack([X_val_raw, X_val_prot])
    y_val = y[val_idx]
    
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain, free_raw_data=True)
    
    # 訓練模型
    model = lgb.train(
        lgb_params,
        dtrain,
        num_boost_round=1000,
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=50) # 減少日誌輸出頻率，保持介面乾淨
        ]
    )
    
    # 預測：注意 predict 之後也要釋放 X_val
    oof_lgb[val_idx] = model.predict(X_val)
    
    # 保存模型 (建議每個 Fold 都存，萬一後面掛了不用重跑)
    model.save_model(f'lgbm_fold_{fold+1}.txt')
    
    # 4. 徹底清理記憶體
    print(f"🧹 Clearing memory for Fold {fold+1}...")
    del dtrain, dval, X_val, X_val_raw, X_val_prot, y_val, model
    gc.collect()
    
    # 給系統 3 秒鐘緩衝，確保作業系統回收記憶體
    time.sleep(3)
    

# 5. 計算分數
print(f"\n🔥 LGBM OOF CV Score: {average_precision_score(y, oof_lgb):.4f}")
np.save('oof_lgb.npy', oof_lgb)
del oof_lgb
gc.collect()

In [ ]:
import xgboost as xgb
from scipy import sparse
import gc

# 1. 統一參數配置
xgb_params = {
    'objective': 'binary:logistic',
    'tree_method': 'hist',      # 大數據必備
    'device': 'cuda',           # 如果有 GPU 請開啟cuda
    'learning_rate': 0.05,
    'max_depth': 6,
    'scale_pos_weight': 4,     # 針對不平衡數據
    'eval_metric': 'aucpr',     # 競賽指標 PR-AUC
    'max_bin': 64,              # 降低記憶體壓力
    'random_state': 42,
    'nthread': -1               # 使用所有 CPU 核心處理數據
}

# 2. 合併特徵矩陣
print("🔗 正在合併特徵矩陣並清理原始零件...")
# 確保 X_fp 和 X_prot 已經定義
X_full = sparse.hstack([X_fp, sparse.csr_matrix(X_prot)]) 
del X_fp, X_prot 
gc.collect()

oof_xgb = np.zeros(len(y))

# 3. 5-Fold 訓練
gkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42) # n folds

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_full, y, groups=groups)):
    print(f"\n🚀 --- Training XGBoost Fold {fold+1} (Sparse Mode) ---")
    
    # --- 訓練集處理：先切片再轉 DMatrix，手動釋放中間變數 ---
    X_tr = X_full[train_idx]
    y_tr = y[train_idx]
    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    del X_tr, y_tr  
    gc.collect()
    
    # --- 驗證集處理 ---
    X_va = X_full[val_idx]
    y_va = y[val_idx]
    dval = xgb.DMatrix(X_va, label=y_va)
    del X_va        # 保持 y_va 稍後計算 OOF
    gc.collect()
    
    # 訓練模型
    watchlist = [(dtrain, 'train'), (dval, 'validation')]
    
    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=1000,
        evals=watchlist,
        early_stopping_rounds=50,
        verbose_eval=50
    )
    
    # 預測並儲存 OOF
    oof_xgb[val_idx] = model.predict(dval)
    
    # 保存模型
    model.save_model(f'xgb_fold_{fold+1}.json')
    
    # 4. 徹底清理
    print(f"🧹 Clearing memory for Fold {fold+1}...")
    del dtrain, dval, model 
    gc.collect()
    
    # 讓系統喘口氣，釋放 IO 緩存
    time.sleep(3)

# --- 5. 總結與存檔 ---
final_score = average_precision_score(y, oof_xgb)
print(f"\n🔥 XGBoost Final OOF CV Score: {final_score:.4f}")

np.save('oof_xgb.npy', oof_xgb)

# 徹底釋放所有數據，準備給 Stacking 使用
del X_full, y, oof_xgb
gc.collect()

In [ ]:
'''
import xgboost as xgb
from scipy import sparse
import gc

xgb_params = {
    'objective': 'binary:logistic',
    'tree_method': 'hist',     # 大數據必備
    'device': 'cuda',          # 如果有 GPU 請開啟
    'learning_rate': 0.05,
    'max_depth': 6,
    'scale_pos_weight': 10,
    'eval_metric': 'aucpr',    # 針對不平衡數據更準確的指標
    'max_bin': 64,
    'random_state': 42
}

oof_xgb = np.zeros(len(y))

# 3. 5-Fold 訓練
for fold, (train_idx, val_idx) in enumerate(gkf.split(X_fp, y, groups=groups)):
    print(f"\n🚀 --- Training XGBoost Fold {fold+1} (Sparse Mode) ---")
    
    # --- 訓練集處理 ---
    # 先合併這一個 Fold 的特徵
    X_train_tmp = sparse.hstack([X_fp[train_idx], sparse.csr_matrix(X_prot[train_idx])])
    
    # 建立 DMatrix，enable_categorical 若有類別特徵可開啟
    # nthread 設定為 -1 使用所有 CPU 核心
    dtrain = xgb.DMatrix(X_train_tmp, label=y[train_idx], nthread=-1)
    
    # 建立完 DMatrix 後立即手動刪除中間矩陣
    del X_train_tmp
    gc.collect() 
    
    # --- 驗證集處理 ---
    X_val_tmp = sparse.hstack([X_fp[val_idx], sparse.csr_matrix(X_prot[val_idx])])
    dval = xgb.DMatrix(X_val_tmp, label=y[val_idx], nthread=-1)
    
    # 訓練模型
    # 使用 evals 來監控驗證集表現
    watchlist = [(dtrain, 'train'), (dval, 'validation')]
    
    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=1000,
        evals=watchlist,
        early_stopping_rounds=50,
        verbose_eval=50
    )
    
    # 預測
    oof_xgb[val_idx] = model.predict(dval)
    
    # 保存模型以防後續崩潰
    model.save_model(f'xgb_fold_{fold+1}.json')
    
    # 4. 徹底清理記憶體
    print(f"🧹 Clearing memory for Fold {fold+1}...")
    # 注意：這裡要刪除的是本次迴圈定義的所有變數
    del dtrain, dval, X_val_tmp, model
    gc.collect()
    
    time.sleep(3)
    

print(f"\n🔥 XGBoost OOF CV Score: {average_precision_score(y, oof_xgb):.4f}")
np.save('oof_xgb.npy', oof_xgb)
del oof_xgb
gc.collect()
'''

In [8]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

class BelkaDataset(Dataset):
    def __init__(self, smiles_seq, protein_ids, protein_nums, labels=None):
        self.smiles_seq = torch.tensor(smiles_seq, dtype=torch.long)
        self.protein_ids = torch.tensor(protein_ids, dtype=torch.long)
        self.protein_nums = torch.tensor(protein_nums, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32) if labels is not None else None

    def __len__(self):
        return len(self.smiles_seq)

    def __getitem__(self, idx):
        if self.labels is not None:
            return self.smiles_seq[idx], self.protein_ids[idx], self.protein_nums[idx], self.labels[idx]
        return self.smiles_seq[idx], self.protein_ids[idx], self.protein_nums[idx]

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        # inputs 是模型輸出的 Raw Logits
        BCE_loss = nn.functional.binary_cross_entropy_with_logits(inputs, targets.view(-1, 1), reduction='none')
        pt = torch.exp(-BCE_loss) # 預測正確的機率
        F_loss = self.alpha * (1 - pt)**self.gamma * BCE_loss
        return F_loss.mean()

In [9]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold
from tqdm import tqdm

# 1.1 定義 SMILES 字符集與編碼器
# 這是在 BELKA 中常見的 token 集合
SMILES_CHARS = ['[PAD]', 'C', 'N', 'O', 'S', 'F', 'Cl', 'Br', 'I', '(', ')', '=', '#', '@', '/', '\\', '[', ']', '1', '2', '3', '4', '5', '6', '7', '8', '9', '0', '+', '-', 'H', 'c', 'n', 'o', 's']
char_to_int = {char: i for i, char in enumerate(SMILES_CHARS)}

def smiles_to_int_seq(smiles, max_len=142):
    """將 SMILES 轉為固定長度的整數序列"""
    seq = [char_to_int.get(c, 0) for c in smiles] # 0 是 [PAD]
    if len(seq) < max_len:
        seq = seq + [0] * (max_len - len(seq)) # Padding
    else:
        seq = seq[:max_len] # Truncating
    return np.array(seq, dtype=np.int32)

# 1.2 定義 PyTorch Dataset (同時加載序列、蛋白 ID 與數值特徵)
class BelkaCNNDataset(Dataset):
    def __init__(self, smiles_seqs, protein_ids, protein_numerics, binds):
        self.smiles_seqs = torch.from_numpy(smiles_seqs).long()
        self.protein_ids = torch.from_numpy(protein_ids).long()
        self.protein_numerics = torch.from_numpy(protein_numerics).float()
        self.binds = torch.from_numpy(binds).float()
        
    def __len__(self):
        return len(self.smiles_seqs)
    
    def __getitem__(self, idx):
        return self.smiles_seqs[idx], self.protein_ids[idx], self.protein_numerics[idx], self.binds[idx]

# 1.3 準備數據 (假設已執行之前修正後的 preprocess_pipeline)
# 這裡需要 SMILES Tokenizer 處理後的序列
df_cnn = pd.read_csv("optimized_train_data.csv", usecols=['molecule_smiles', 'protein_id','protein_name', 'buildingblock1_smiles', 'binds'])
# 注意：你需要補上之前 extract_protein_features 產出的 3 個標準化數值特徵
df, prot_num_cols = extract_protein_features(df_cnn)
print("🚀 正在執行 SMILES Tokenization (這可能需要幾分鐘)...")

# 使用 list comprehension 並立即轉為 int8/int16 以節省 50-75% 記憶體
smiles_seqs = np.array([smiles_to_int_seq(s) for s in tqdm(df['molecule_smiles'])], dtype=np.int8)

print("🚀 正在 Tokenizing SMILES...")
protein_ids = df['protein_id'].values.astype(np.int8)
protein_numerics = df[prot_num_cols].values.astype(np.float32)
binds = df['binds'].values.astype(np.float32)
groups = df['buildingblock1_smiles'].values # 用於 GroupKFold

print(f"✅ CNN 數據就緒: Seqs={smiles_seqs.shape}")
print(f"佔用記憶體: {smiles_seqs.nbytes / 1024**2:.2f} MB")

🚀 正在執行 SMILES Tokenization (這可能需要幾分鐘)...


100%|██████████| 6378492/6378492 [01:43<00:00, 61910.20it/s]


🚀 正在 Tokenizing SMILES...
✅ CNN 數據就緒: Seqs=(6378492, 142)
佔用記憶體: 863.79 MB


In [ ]:
'''
from sklearn.metrics import average_precision_score
import gc

# 設定 GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 3.1 訓練與驗證迴圈
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for seqs, p_ids, p_nums, y in loader:
        seqs, p_ids, p_nums, y = seqs.to(device), p_ids.to(device), p_nums.to(device), y.to(device).unsqueeze(1)
        optimizer.zero_grad()
        out = model(seqs, p_ids, p_nums)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate_oof(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for seqs, p_ids, p_nums, y in loader:
            seqs, p_ids, p_nums, y = seqs.to(device), p_ids.to(device), p_nums.to(device), y.to(device)
            out = model(seqs, p_ids, p_nums)
            y_pred.append(torch.sigmoid(out).cpu().numpy()) # 轉為機率
            y_true.append(y.cpu().numpy())
    return np.concatenate(y_pred), np.concatenate(y_true)

# 3.2 執行 5-Fold GroupKFold
gkf = GroupKFold(n_splits=3)

# 它是一個 DataFrame，包含 'prot_MW_kDa', 'prot_pI', 'prot_gravy' 這三欄
prot_num_features = df[prot_num_cols]

# 轉換為 numpy 矩陣供 Dataset 使用 
prot_num_matrix = prot_num_features.values

# 這裡將存儲 CNN 對完整數據集的 Out-of-fold 預測值 (Stacking 用)
oof_cnn = np.zeros(len(smiles_seqs))

vocab_size = len(SMILES_CHARS)
criterion = FocalLoss(alpha=0.25, gamma=2.0)

for fold, (train_idx, val_idx) in enumerate(gkf.split(smiles_seqs, binds, groups=groups)):
    print(f"--- CNN Training Fold {fold+1} ---")

    # 建立 Dataset 與 DataLoader (調整 Batch Size 以適應 GPU 記憶體)
    train_ds = BelkaCNNDataset(smiles_seqs[train_idx], protein_ids[train_idx], prot_num_matrix[train_idx], binds[train_idx])
    val_ds = BelkaCNNDataset(smiles_seqs[val_idx], protein_ids[val_idx], prot_num_matrix[val_idx], binds[val_idx])
    train_loader = DataLoader(train_ds, batch_size=1024, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=2048, shuffle=False)
    
    # 模型初始化
    model = BelkaFusionModel(vocab_size).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    # 可選：加入學習率調度器
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

    # 簡單的 Early Stopping
    best_val_aucpr = -1
    for epoch in range(15): # 通常 CNN 在此數據上 10-20 Epoch 收斂
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        y_val_pred, y_val_true = evaluate_oof(model, val_loader, device)
        
        # 使用 aucpr (PR-AUC) 作為監控指標，這對不平衡數據更準確
        val_aucpr = average_precision_score(y_val_true, y_val_pred)
        print(f"Epoch {epoch+1} - Loss: {train_loss:.4f}, Val AUCPR: {val_aucpr:.4f}")
        
        # 更新學習率
        scheduler.step(val_aucpr)
        
        # 儲存最佳模型預測
        if val_aucpr > best_val_aucpr:
            best_val_aucpr = val_aucpr
            oof_cnn[val_idx] = y_val_pred.flatten()
            torch.save(model.state_dict(), f'cnn_fold_{fold+1}.pt')

    del train_loader, val_loader, train_ds, val_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print(f"✅ CNN OOF CV Score: {average_precision_score(binds, oof_cnn):.4f}")
# 將 CNN 的預測結果存起來，用於 Stacking
np.save('oof_cnn.npy', oof_cnn)
del oof_cnn

# 徹底刪除 CNN 相關的大型對象
del smiles_seqs, protein_ids, protein_numerics, binds
# 如果有 Dataset 或 Loader 也一併刪除
if 'train_ds' in locals(): del train_ds, val_ds, train_loader, val_loader

gc.collect()
torch.cuda.empty_cache() # 釋放顯存

print("🧹 記憶體已清空...")
'''

In [ ]:
@torch.no_grad()
def update_bn_custom(loader, model, device):
    model.train() # 必須在 train 模式下 BN 才會更新統計量
    for seqs, p_ids, p_nums, _ in loader: 
        seqs = seqs.to(device)
        p_ids = p_ids.to(device)
        p_nums = p_nums.to(device)
        # 執行 Forward pass，SWA 會自動在後台更新模組內的 BN 統計量
        model(seqs, p_ids, p_nums)

In [ ]:
import gc
from sklearn.metrics import average_precision_score
from torch.optim.lr_scheduler import LinearLR, ReduceLROnPlateau
from torch.optim.swa_utils import AveragedModel, SWALR
import torch.nn.functional as F

# 設定 GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 3.1 訓練與驗證迴圈
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for seqs, p_ids, p_nums, y in loader:
        seqs, p_ids, p_nums, y = seqs.to(device), p_ids.to(device), p_nums.to(device), y.to(device).unsqueeze(1)
        optimizer.zero_grad()
        out = model(seqs, p_ids, p_nums)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate_oof(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for seqs, p_ids, p_nums, y in loader:
            seqs, p_ids, p_nums, y = seqs.to(device), p_ids.to(device), p_nums.to(device), y.to(device)
            out = model(seqs, p_ids, p_nums)
            y_pred.append(torch.sigmoid(out).detach().cpu().numpy()) # 轉為機率
            y_true.append(y.cpu().numpy())
    return np.concatenate(y_pred), np.concatenate(y_true)
    
def smooth_focal_loss(outputs, targets, criterion, epsilon=0.05):
    # Label Smoothing: 0 -> 0.025, 1 -> 0.975 (當 epsilon=0.05)
    targets = targets.float()
    targets = targets * (1 - epsilon) + epsilon / 2
    return criterion(outputs, targets)
    
# 執行 5-Fold StratifiedGroupKFold 確保「蛋白質不重疊」，並且「每個 Fold 的正負樣本比例 (binds) 盡可能一致」
gkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# 它是一個 DataFrame，包含 'prot_MW_kDa', 'prot_pI', 'prot_gravy' 這三欄
prot_num_features = df[prot_num_cols]

# 轉換為 numpy 矩陣供 Dataset 使用 
prot_num_matrix = prot_num_features.values

# 這裡將存儲 CNN 對完整數據集的 Out-of-fold 預測值 (Stacking 用)
oof_cnn = np.zeros(len(smiles_seqs))

vocab_size = len(SMILES_CHARS)
criterion = FocalLoss(alpha=0.25, gamma=2.0)

for fold, (train_idx, val_idx) in enumerate(gkf.split(smiles_seqs, binds, groups=groups)):
    print(f"\n--- CNN Training Fold {fold+1} (Advanced Mode) ---")
    
    # 建立 Dataset 與 DataLoader (調整 Batch Size 以適應 GPU 記憶體)
    train_ds = BelkaCNNDataset(smiles_seqs[train_idx], protein_ids[train_idx], prot_num_matrix[train_idx], binds[train_idx])
    val_ds = BelkaCNNDataset(smiles_seqs[val_idx], protein_ids[val_idx], prot_num_matrix[val_idx], binds[val_idx])
    train_loader = DataLoader(train_ds, batch_size=1024, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=2048, shuffle=False)
    
    model = BelkaFusionModel(vocab_size).to(device)
    if torch.cuda.device_count() > 1:
        print(f"🔥 偵測到 {torch.cuda.device_count()} 張 GPU，啟動 DataParallel 雙卡平行運算！")
        model = torch.nn.DataParallel(model)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-2)
    
    # --- 2. 初始化 SWA 元件 ---
    # AveragedModel 會幫你儲存權重的移動平均
    swa_model = AveragedModel(model)
    # SWA 通常在訓練後期開啟，這裡設定在第 10 個 Epoch 進入 SWA 模式
    swa_start = 10 
    swa_scheduler = SWALR(optimizer, swa_lr=0.00005) # SWA 階段使用穩定的 LR
    
    warmup_scheduler = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=len(train_loader))
    main_scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

    best_val_aucpr = -1
    early_stop_counter = 0

    for epoch in range(15):
        model.train()
        total_loss = 0
        
        for batch_idx, (seqs, p_ids, p_nums, y) in enumerate(train_loader):
            seqs, p_ids, p_nums, y = seqs.to(device), p_ids.to(device), p_nums.to(device), y.to(device).unsqueeze(1)
            
            optimizer.zero_grad()
            out = model(seqs, p_ids, p_nums)
            
            # ✅ 加入標籤平滑 (Label Smoothing)
            loss = smooth_focal_loss(out, y, criterion, epsilon=0.05)
            
            loss.backward()

            # ✅ 加入梯度裁剪 (Gradient Clipping) - 防止 Focal Loss 梯度噴發
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            if epoch == 0:
                warmup_scheduler.step()
            
            total_loss += loss.item()
        
        # 計算本回合的平均 Training Loss
        train_loss = total_loss / len(train_loader)
        
        # ================================
        # 2. 驗證階段 (Eval) - 🚨 永遠只用 model
        # ================================
        model.eval() 
        
        with torch.no_grad():
            y_val_pred, y_val_true = evaluate_oof(model, val_loader, device)
        val_aucpr = average_precision_score(y_val_true, y_val_pred)
        
        # ================================
        # 3. 學習率與 SWA 權重更新
        # ================================
        if epoch >= swa_start:
            # SWA 躲在幕後，把剛剛訓練完的 model 權重抄寫下來平均
            swa_model.update_parameters(model)
            swa_scheduler.step()
        else:
            main_scheduler.step(val_aucpr)

        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1} - Loss: {train_loss:.4f}, Val AUCPR: {val_aucpr:.4f}, LR: {current_lr:.6f} {'(SWA Active)' if epoch >= swa_start else ''}")

        # ================================
        # 4. 儲存與早停機制 (迴圈內只存 model)
        # ================================
        if val_aucpr > best_val_aucpr:
            best_val_aucpr = val_aucpr
            early_stop_counter = 0
            oof_cnn[val_idx] = y_val_pred.flatten()
            
            # 剝洋蔥存檔法：不管 SWA，我們只確保當下最強的 model 被存下來當保險
            if isinstance(model, torch.nn.DataParallel):
                final_state_dict = model.module.state_dict()
            else:
                final_state_dict = model.state_dict()
                
            torch.save(final_state_dict, f'cnn_fold_{fold+1}.pt')
            print(f"🌟 Best model saved!")
            
        elif epoch < swa_start:
            early_stop_counter += 1
            if early_stop_counter >= 15: 
                print(f"🛑 Early Stopping triggered!")
                break

    # ================================
    # 5. 大結局：SWA 終極結算與覆蓋存檔
    # ================================
    if epoch >= swa_start: 
        print("🔄 訓練結束，正在為 SWA 模型更新 Batch Norm 統計數據...")
        # 讓 swa_model 重新看過一遍 train_loader，算真實的 BN 數據
        update_bn_custom(train_loader, swa_model, device=device)
        
        # 🌟 核心修正：剝掉 SWA 和 DataParallel 的外殼，取得最純淨的終極權重！
        if isinstance(swa_model.module, torch.nn.DataParallel):
            final_swa_state = swa_model.module.module.state_dict()
        else:
            final_swa_state = swa_model.module.state_dict()
            
        # 覆蓋掉迴圈中存的那個暫時的 model，換成最強的完成體
        torch.save(final_swa_state, f'cnn_fold_{fold+1}.pt')
        print("✅ 最終 SWA 模型 (含完整 BN 數據) 已覆蓋儲存！")
    # torch.optim.swa_utils.update_bn(train_loader, swa_model, device=device)

    try:
        del train_ds, val_ds, train_loader, val_loader
    except NameError:
        pass

    try:
        del model, swa_model, optimizer, warmup_scheduler, main_scheduler, swa_scheduler
    except NameError:
        pass

    try:
        del model_to_save, core_model, final_state_dict, final_swa_state
    except NameError:
        pass
        
    gc.collect()
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(3)


print(f"✅ CNN OOF CV Score: {average_precision_score(binds, oof_cnn):.4f}")
# 將 CNN 的預測結果存起來，用於 Stacking
np.save('oof_cnn.npy', oof_cnn)
del oof_cnn

# 徹底刪除 CNN 相關的大型對象
del smiles_seqs, protein_ids, protein_numerics, binds
# 如果有 Dataset 或 Loader 也一併刪除
try:
    del train_ds, val_ds, train_loader, val_loader
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache() # 釋放顯存

print("🧹 記憶體已清空...")

In [ ]:
oof_lgb = np.load('oof_lgb.npy')
oof_xgb = np.load('oof_xgb.npy')
oof_cnn = np.load('oof_cnn.npy')

# 準備 Meta-model 的訓練數據
X_meta = np.column_stack([oof_lgb, oof_xgb, oof_cnn])
print(X_meta.shape[0])

載入模型

方案一

In [ ]:
'''
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import average_precision_score

# 1. 載入第一層模型的 OOF 預測結果 (Level 0 outputs)
oof_lgb = np.load('/kaggle/input/datasets/t8101349/oof-pred/oof_lgb.npy')
oof_xgb = np.load('/kaggle/input/datasets/t8101349/oof-pred/oof_xgb.npy')
oof_cnn = np.load('/kaggle/input/datasets/t8101349/oof-pred/oof_cnn.npy')

# 載入真實標籤
y_true = pd.read_csv("optimized_train_data.csv", usecols=['binds']).values[:n_samples]

# 2. 組合特徵矩陣
# 每一列代表一個模型的預測機率
X_meta = np.column_stack([oof_lgb, oof_xgb, oof_cnn])

# 3. 訓練 Meta-Model (這裡使用 Ridge，因為它對權重的分配很穩定)
meta_model = Ridge(alpha=1.0)
meta_model.fit(X_meta, y_true)

# 4. 查看模型權重 (這能告訴你哪個模型最有用)
weights = meta_model.coef_
print(f"Stacking Weights -> LGBM: {weights[0]:.3f}, XGB: {weights[1]:.3f}, CNN: {weights[2]:.3f}")

# 5. 計算最終 Stacking OOF 分數
final_oof_pred = meta_model.predict(X_meta)
# 限制在 0~1 之間
final_oof_pred = np.clip(final_oof_pred, 0, 1)

final_score = average_precision_score(y_true, final_oof_pred)
print(f"Final Stacking CV Score (mAP): {final_score:.4f}")
'''

In [14]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

# 1. 載入 OOF (確認資料量與順序完全一致)
oof_lgb = np.load('/kaggle/input/datasets/t8101349/oof-pred/oof_lgb.npy')
oof_xgb = np.load('/kaggle/input/datasets/t8101349/oof-pred/oof_xgb.npy')
oof_cnn = np.load('/kaggle/input/datasets/t8101349/oof-pred/oof_cnn.npy')

# 2. 載入真實標籤 (🌟 修正維度問題，攤平為 1D)
y_true = pd.read_csv("optimized_train_data.csv", usecols=['binds'])['binds'].values

# 組合特徵矩陣
X_meta = np.column_stack([oof_lgb, oof_xgb, oof_cnn])

print("🚀 啟動 Meta-Model 5-Fold 交叉驗證...")

# 準備容器
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
meta_oof_preds = np.zeros(len(y_true))
meta_models = []

# 3. 執行 Meta-CV
for fold, (train_idx, val_idx) in enumerate(skf.split(X_meta, y_true)):
    X_tr, y_tr = X_meta[train_idx], y_true[train_idx]
    X_va, y_va = X_meta[val_idx], y_true[val_idx]
    
    # 🌟 改用 LogisticRegression 處理機率融合
    model = LogisticRegression(max_iter=1000)
    model.fit(X_tr, y_tr)
    
    # 預測驗證集機率 (注意：LogisticRegression 要用 predict_proba 取第 1 類的機率)
    meta_oof_preds[val_idx] = model.predict_proba(X_va)[:, 1]
    
    # 儲存模型供後續 Inference 測試集使用
    meta_models.append(model)
    
    # 印出單折的權重，觀察各模型貢獻度是否穩定
    weights = model.coef_[0]
    print(f"Fold {fold+1} Weights -> LGBM: {weights[0]:.3f}, XGB: {weights[1]:.3f}, CNN: {weights[2]:.3f}")

# 4. 計算最終真實的 Stacking CV 分數
final_real_score = average_precision_score(y_true, meta_oof_preds)
print(f"\n🔥 真實最終 Stacking CV 分數 (mAP): {final_real_score:.4f}")

# 5. (選項) 訓練一個「吃全部資料」的最終 Meta-Model 準備去打 Test Set
final_meta_model = LogisticRegression(max_iter=1000)
final_meta_model.fit(X_meta, y_true)
final_weights = final_meta_model.coef_[0]
print(f"\n👑 全局最終權重 -> LGBM: {final_weights[0]:.3f}, XGB: {final_weights[1]:.3f}, CNN: {final_weights[2]:.3f}")

🚀 啟動 Meta-Model 5-Fold 交叉驗證...
Fold 1 Weights -> LGBM: 3.406, XGB: 2.651, CNN: 2.739
Fold 2 Weights -> LGBM: 3.392, XGB: 2.666, CNN: 2.742
Fold 3 Weights -> LGBM: 3.404, XGB: 2.655, CNN: 2.744
Fold 4 Weights -> LGBM: 3.403, XGB: 2.656, CNN: 2.741
Fold 5 Weights -> LGBM: 3.402, XGB: 2.654, CNN: 2.739
Fold 6 Weights -> LGBM: 3.404, XGB: 2.655, CNN: 2.737
Fold 7 Weights -> LGBM: 3.395, XGB: 2.663, CNN: 2.740
Fold 8 Weights -> LGBM: 3.397, XGB: 2.658, CNN: 2.743
Fold 9 Weights -> LGBM: 3.395, XGB: 2.663, CNN: 2.739
Fold 10 Weights -> LGBM: 3.402, XGB: 2.655, CNN: 2.737

🔥 真實最終 Stacking CV 分數 (mAP): 0.9103

👑 全局最終權重 -> LGBM: 3.400, XGB: 2.658, CNN: 2.740


In [15]:
import numpy as np
import pandas as pd
from scipy import sparse
import lightgbm as lgb
import xgboost as xgb

input_file = "/kaggle/input/leash-BELKA/test.csv"
df_test = pd.read_csv(input_file)
df, prot_num_cols = extract_protein_features(df_test)
num_samples = len(df_test)

# 1. 建立儲存清單
sparse_list = []
batch_size = 50000

print(f"🚀 開始以稀疏格式處理 {len(df_test)} 筆數據...")

for i in tqdm(range(0, len(df_test), batch_size)):
    end_i = min(i + batch_size, len(df_test))
    batch_smiles = df_test['molecule_smiles'].iloc[i:end_i].values
    
    # 生成指紋 (預設是密集矩陣)
    batch_fps = smiles_to_fp_fast(batch_smiles)
    
    # 轉換為 CSR 稀疏格式
    # 這會瞬間釋放掉 0 所佔用的空間
    batch_fps_sparse = sparse.csr_matrix(batch_fps.astype('uint8'))
    sparse_list.append(batch_fps_sparse)

# 2. 合併所有分塊
test_fp_sparse = sparse.vstack(sparse_list)

# 3. 儲存成 .npz 格式 
sparse.save_npz('test_fp_sparse.npz', test_fp_sparse)

# 儲存蛋白質特徵與標籤 (這兩個很小，直接存 Parquet)
df[prot_num_cols].to_parquet('test_prot_features.parquet')

del sparse_list, test_fp_sparse, df_test
gc.collect()


🚀 開始以稀疏格式處理 1674896 筆數據...


100%|██████████| 34/34 [14:13<00:00, 25.10s/it]


236

In [16]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# --- A. GBDT 推理 ---
# 合併測試集的指紋與蛋白質數值特徵
test_fp_sparse = sparse.load_npz('test_fp_sparse.npz')
X_test_prot = pd.read_parquet('test_prot_features.parquet').values
X_test_gbdt = sparse.hstack([test_fp_sparse, sparse.csr_matrix(X_test_prot)])
print(f"✅ X_test_gbdt 準備完成！形狀: {X_test_gbdt.shape}")

print("🔮 正在進行 LGBM 與 XGBoost 推理...")
# 準備空陣列來裝預測結果
preds_lgb = np.zeros(X_test_gbdt.shape[0])
preds_xgb = np.zeros(X_test_gbdt.shape[0])

# XGBoost 需要先轉換資料格式
dtest_xgb = xgb.DMatrix(X_test_gbdt)

# --- 載入 5 Fold 模型並平均預測 ---
for fold in range(1, 6):
    # 1. 載入 LGBM (.txt)
    lgb_model = lgb.Booster(model_file=f'/kaggle/input/models/t8101349/prime-model/pytorch/default/1/lgbm_fold_{fold}.txt')
    preds_lgb += lgb_model.predict(X_test_gbdt) / 5.0
    
    # 2. 載入 XGBoost (.json)
    xgb_model = xgb.Booster()
    xgb_model.load_model(f'/kaggle/input/models/t8101349/prime-model/pytorch/default/1/xgb_fold_{fold}.json')
    preds_xgb += xgb_model.predict(dtest_xgb) / 5.0

print("✅ 樹模型 5-Fold 推理完成！")




✅ X_test_gbdt 準備完成！形狀: (1674896, 2051)
🔮 正在進行 LGBM 與 XGBoost 推理...
✅ 樹模型 5-Fold 推理完成！


In [17]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import TensorDataset

print("📥 正在重新讀取 Test Set (只讀取需要的欄位以節省記憶體)...")
df_test_cnn = pd.read_csv("/kaggle/input/leash-BELKA/test.csv", usecols=['molecule_smiles', 'protein_name'])

# ==========================================
# 1. 執行蛋白質特徵萃取 (數值化)
# ==========================================
print("🧬 正在萃取蛋白質數值特徵...")
# 這裡會跑你寫的特徵工程函數，取得 df 和數值欄位列表 prot_num_cols
df, prot_num_cols = extract_protein_features(df_test_cnn)

# 取得蛋白質數值特徵矩陣
test_protein_numerics = df[prot_num_cols].values.astype(np.float32)

# ==========================================
# 2. 轉換 Protein IDs (sEH:0, BRD4:1, HSA:2)
# ==========================================
print("🧬 正在轉換蛋白質名稱為 ID...")
protein_map = {'sEH': 0, 'BRD4': 1, 'HSA': 2}
test_protein_ids = df['protein_name'].map(protein_map).values.astype(np.int8)

# ==========================================
# 3. 轉換 SMILES 序列 (Tokens)
# ==========================================
print("🔡 正在執行 SMILES Tokenization (這可能需要幾分鐘)...")
# 使用 int8 可以省下極大量記憶體！
# 🚨 請確保 smiles_to_int_seq 輸出的長度跟訓練時一模一樣 (例如 max_len=142)
test_smiles_seqs = np.array([smiles_to_int_seq(s) for s in tqdm(df['molecule_smiles'])], dtype=np.int8)

print(f"✅ CNN 數據就緒: Seqs形狀={test_smiles_seqs.shape}")

# 確認你的字典大小 (CNN 模型架構需要用到)
vocab_size = len(SMILES_CHARS)
print(f"📖 字典大小 (vocab_size): {vocab_size}")

# ==========================================
# 4. 建立 CNN 專屬的 DataLoader
# ==========================================
print("📦 正在打包 CNN Dataset...")
cnn_dataset = TensorDataset(
    torch.from_numpy(test_smiles_seqs).long(),
    torch.from_numpy(test_protein_ids).long(),
    torch.from_numpy(test_protein_numerics).float()
)

# 徹底清理記憶體
del df_test_cnn, df, test_smiles_seqs, test_protein_ids, test_protein_numerics
gc.collect()

print("✅ CNN 測試集準備完成，可以開始推論了！")


📥 正在重新讀取 Test Set (只讀取需要的欄位以節省記憶體)...
🧬 正在萃取蛋白質數值特徵...
🧬 正在轉換蛋白質名稱為 ID...
🔡 正在執行 SMILES Tokenization (這可能需要幾分鐘)...


100%|██████████| 1674896/1674896 [00:24<00:00, 69444.82it/s]


✅ CNN 數據就緒: Seqs形狀=(1674896, 142)
📖 字典大小 (vocab_size): 35
📦 正在打包 CNN Dataset...
✅ CNN 測試集準備完成，可以開始推論了！


In [18]:
# --- B. CNN 推理 ---
print("🔮 正在進行 CNN 推理 (Batch Mode)...")

# 推論時不需要計算梯度，可以把 batch_size 開大一點加速 (例如 2048)
cnn_loader = DataLoader(cnn_dataset, batch_size=2048, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 準備空陣列來裝 5 折平均結果
preds_cnn = np.zeros(len(cnn_dataset))

# 建立一個空的模型架構 (記得傳入正確的 vocab_size)
cnn_model = BelkaFusionModel(vocab_size=vocab_size).to(device)

for fold in range(1, 6):
    print(f"📥 載入 CNN Fold {fold}...")
    
    # 讀取權重並切換為推論模式
    cnn_model.load_state_dict(torch.load(f'/kaggle/input/models/t8101349/prime-model/pytorch/default/1/cnn_fold_{fold}.pt', map_location=device))
    cnn_model.eval() 
    
    fold_preds = []
    
    # 關閉梯度計算
    with torch.no_grad(): 
        for seqs, p_ids, p_nums in cnn_loader:
            seqs, p_ids, p_nums = seqs.to(device), p_ids.to(device), p_nums.to(device)
            out = cnn_model(seqs, p_ids, p_nums)
            fold_preds.extend(out.cpu().numpy().flatten())
            
    # 🚨 關鍵 2：把這個 fold 的預測值變成 Numpy array 後，除以 5 加進總和裡
    preds_cnn += np.array(fold_preds) / 5.0

print(f"✅ CNN 推理完成，形狀校正為: {preds_cnn.shape}")

🔮 正在進行 CNN 推理 (Batch Mode)...
📥 載入 CNN Fold 1...
📥 載入 CNN Fold 2...
📥 載入 CNN Fold 3...
📥 載入 CNN Fold 4...
📥 載入 CNN Fold 5...
✅ CNN 推理完成，形狀校正為: (1674896,)


In [39]:
# 將預測結果對齊並合併為矩陣 (Shape: [1600000, 3])
X_meta_test = np.column_stack([
    preds_lgb, 
    preds_xgb, 
    preds_cnn
])

print(f"📊 Meta-feature shape: {X_meta_test.shape}")

# final_preds = final_meta_model.predict(X_meta_test)
# final_preds = np.clip(final_preds, 0, 1)

final_preds = final_meta_model.predict_proba(X_meta_test)[:, 1]
final_preds = np.clip(final_preds, 0, 1)

📊 Meta-feature shape: (1674896, 3)


In [40]:
import gc

try:
    del X_meta_test, X_test_gbdt, smiles_seqs, test_fp_sparse
except:
    pass
gc.collect()

print("📝 正在封裝 Submission 檔案...")

test_ids = pd.read_parquet("/kaggle/input/leash-BELKA/test.parquet", columns=['id'])['id'].values

# 建立最終 DataFrame
submission = pd.DataFrame({
    'id': test_ids,
    'binds': final_preds
})

# 存檔
submission.to_csv('submission.csv', index=False)

print("---")
print("提交檔 'submission.csv' 已準備就緒。")
# 最後檢查行數是否正確
sample_sub = pd.read_csv("/kaggle/input/leash-BELKA/sample_submission.csv", usecols=['id'])
if len(submission) == len(sample_sub):
    print(f"✅ 行數檢查通過: {len(submission)} 筆數據")
else:
    print(f"❌ 警告：行數不符！提交檔: {len(submission)}, 範本: {len(sample_sub)}")

📝 正在封裝 Submission 檔案...
---
提交檔 'submission.csv' 已準備就緒。
✅ 行數檢查通過: 1674896 筆數據


In [23]:
submission_lgb = pd.DataFrame({
    'id': test_ids,
    'binds': preds_lgb
})

# 存檔
submission_lgb.to_csv('submission_lgb.csv', index=False)

print("---")
print("提交檔 'submission_lgb.csv' 已準備就緒。")

if len(submission) == len(sample_sub):
    print(f"✅ 行數檢查通過: {len(submission)} 筆數據")
else:
    print(f"❌ 警告：行數不符！提交檔: {len(submission)}, 範本: {len(sample_sub)}")

---
提交檔 'submission_lgb.csv' 已準備就緒。
✅ 行數檢查通過: 1674896 筆數據


方案二

In [30]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import average_precision_score

# 1. 載入第一層模型的 OOF 預測 (Level 0 outputs)
oof_lgb = np.load('/kaggle/input/datasets/t8101349/oof-pred/oof_lgb.npy')
oof_xgb = np.load('/kaggle/input/datasets/t8101349/oof-pred/oof_xgb.npy')
oof_cnn = np.load('/kaggle/input/datasets/t8101349/oof-pred/oof_cnn.npy')

# 2. 載入標籤與蛋白質 ID (為了讓 Meta-model 知道現在是哪個蛋白)
df_meta = pd.read_csv("optimized_train_data.csv", usecols=['protein_name', 'binds'])
protein_map = {'sEH': 0, 'BRD4': 1, 'HSA': 2}
train_protein_ids = df_meta['protein_name'].map(protein_map).values.astype(np.int8)

y_true = df_meta['binds'].values


# 3. 建立 Meta 特徵矩陣
# 我們將 3 個模型的預測值 + 蛋白質 ID 組合成新的訓練集
X_meta = pd.DataFrame({
    'pred_lgb': oof_lgb,
    'pred_xgb': oof_xgb,
    'pred_cnn': oof_cnn,
    'protein_id': train_protein_ids
})
X_meta['protein_id'] = X_meta['protein_id'].astype('category')

print(f"✅ Meta-model 訓練集準備完成，形狀: {X_meta.shape}")

✅ Meta-model 訓練集準備完成，形狀: (6378492, 4)


In [ ]:
'''
# 設定 Meta-model 參數
# 注意：這裡使用 Regressor，因為我們處理的是機率值的回歸與修正
meta_params = {
    'objective': 'regression', # 或者用 binary，但 regression 在融合機率時通常更平滑
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'max_depth': 3,            # 深度要小，防止二次過擬合
    'learning_rate': 0.05,
    'n_estimators': 100,
    'min_child_samples': 50,
    'random_state': 42,
    'verbose': -1
}

# 訓練 Meta-model
print("🚀 正在訓練 Meta-model (LGBM)...")
reg_meta = lgb.LGBMRegressor(**meta_params)
reg_meta.fit(X_meta, y_true)

# 預測並評估
final_oof_preds = reg_meta.predict(X_meta)
# 修正數值範圍
final_oof_preds = np.clip(final_oof_preds, 0, 1)

final_score = average_precision_score(y_true, final_oof_preds)
print(f"🔥 最終 Stacking CV 分數 (mAP): {final_score:.4f}")

# 查看特徵重要性
importance = pd.DataFrame({
    'feature': X_meta.columns,
    'importance': reg_meta.feature_importances_
}).sort_values('importance', ascending=False)
print("\n📊 模型貢獻度分析:")
print(importance)
'''

In [31]:
meta_params = {
    'objective': 'regression', # 或者用 binary，但 regression 在融合機率時通常更平滑
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'max_depth': 3,            # 深度要小，防止二次過擬合
    'learning_rate': 0.05,
    'n_estimators': 100,
    'min_child_samples': 50,
    'random_state': 42,
    'verbose': -1
}

print("🚀 啟動 Meta-model (LGBM) 5-Fold 交叉驗證...")

# 建立儲存 Meta-OOF 的陣列
meta_oof_preds = np.zeros(len(X_meta))
meta_models = [] # 用來存 5 個練好的 Meta-model

# 使用普通的 StratifiedKFold 即可 (因為底層已經解決了 Group 洩漏問題)
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_meta, y_true)):
    print(f"\n--- Meta-Model Training Fold {fold+1} ---")
    
    X_train, y_train = X_meta.iloc[train_idx], y_true[train_idx]
    X_val, y_val = X_meta.iloc[val_idx], y_true[val_idx]
    
    # 訓練 Meta-model
    reg_meta = lgb.LGBMRegressor(**meta_params)
    reg_meta.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)]
    )
    
    # 預測驗證集
    val_preds = reg_meta.predict(X_val)
    meta_oof_preds[val_idx] = np.clip(val_preds, 0, 1)
    
    # 計算單折分數
    fold_score = average_precision_score(y_val, meta_oof_preds[val_idx])
    print(f"Fold {fold+1} Meta-AUCPR: {fold_score:.4f}")
    
    # 儲存這個 fold 的 Meta-model
    reg_meta.booster_.save_model(f'meta_lgbm_fold_{fold+1}.txt')
    meta_models.append(reg_meta)

# 計算最終真實的 Stacking CV 分數
final_real_score = average_precision_score(y_true, meta_oof_preds)
print(f"\n🔥 真實最終 Stacking CV 分數 (mAP): {final_real_score:.4f}")

# 看看誰是老大哥 (特徵重要性平均)
importances = np.mean([model.feature_importances_ for model in meta_models], axis=0)
importance_df = pd.DataFrame({
    'feature': X_meta.columns,
    'importance': importances
}).sort_values('importance', ascending=False)
print("\n📊 模型貢獻度分析:")
print(importance_df)

🚀 啟動 Meta-model (LGBM) 5-Fold 交叉驗證...

--- Meta-Model Training Fold 1 ---
Fold 1 Meta-AUCPR: 0.9261

--- Meta-Model Training Fold 2 ---
Fold 2 Meta-AUCPR: 0.9264

--- Meta-Model Training Fold 3 ---
Fold 3 Meta-AUCPR: 0.9249

--- Meta-Model Training Fold 4 ---
Fold 4 Meta-AUCPR: 0.9253

--- Meta-Model Training Fold 5 ---
Fold 5 Meta-AUCPR: 0.9268

--- Meta-Model Training Fold 6 ---
Fold 6 Meta-AUCPR: 0.9262

--- Meta-Model Training Fold 7 ---
Fold 7 Meta-AUCPR: 0.9261

--- Meta-Model Training Fold 8 ---
Fold 8 Meta-AUCPR: 0.9260

--- Meta-Model Training Fold 9 ---
Fold 9 Meta-AUCPR: 0.9260

--- Meta-Model Training Fold 10 ---
Fold 10 Meta-AUCPR: 0.9259

🔥 真實最終 Stacking CV 分數 (mAP): 0.9260

📊 模型貢獻度分析:
      feature  importance
0    pred_lgb       228.9
2    pred_cnn       212.6
3  protein_id       129.3
1    pred_xgb       129.2


In [ ]:
input_file = "/kaggle/input/leash-BELKA/test.csv"
df_test = pd.read_csv(input_file)
df, prot_num_cols = extract_protein_features(df_test)
num_samples = len(df_test)

# 1. 建立儲存清單
sparse_list = []
batch_size = 50000

print(f"🚀 開始以稀疏格式處理 {len(df_test)} 筆數據...")

for i in tqdm(range(0, len(df_test), batch_size)):
    end_i = min(i + batch_size, len(df_test))
    batch_smiles = df_test['molecule_smiles'].iloc[i:end_i].values
    
    # 生成指紋 (預設是密集矩陣)
    batch_fps = smiles_to_fp_fast(batch_smiles)
    
    # 轉換為 CSR 稀疏格式
    # 這會瞬間釋放掉 0 所佔用的空間
    batch_fps_sparse = sparse.csr_matrix(batch_fps.astype('uint8'))
    sparse_list.append(batch_fps_sparse)

# 2. 合併所有分塊
test_fp_sparse = sparse.vstack(sparse_list)

# 3. 儲存成 .npz 格式 
sparse.save_npz('test_fp_sparse.npz', test_fp_sparse)

# 儲存蛋白質特徵與標籤 (這兩個很小，直接存 Parquet)
df[prot_num_cols].to_parquet('test_prot_features.parquet')

del sparse_list, test_fp_sparse, df_test
gc.collect()


In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# --- A. GBDT 推理 ---
# 合併測試集的指紋與蛋白質數值特徵
X_test_prot = pd.read_parquet('test_prot_features.parquet').values
X_test_gbdt = sparse.hstack([test_fp_sparse, sparse.csr_matrix(X_test_prot)])

print("🔮 正在進行 LGBM 與 XGBoost 推理...")
# 準備空陣列來裝預測結果
preds_lgb = np.zeros(X_test_gbdt.shape[0])
preds_xgb = np.zeros(X_test_gbdt.shape[0])

# XGBoost 需要先轉換資料格式
dtest_xgb = xgb.DMatrix(X_test_gbdt)

# --- 載入 5 Fold 模型並平均預測 ---
for fold in range(1, 6):
    # 1. 載入 LGBM (.txt)
    lgb_model = lgb.Booster(model_file=f'/kaggle/input/models/t8101349/prime-model/pytorch/default/1/lgbm_fold_{fold}.txt')
    preds_lgb += lgb_model.predict(X_test_gbdt) / 5.0
    
    # 2. 載入 XGBoost (.json)
    xgb_model = xgb.Booster()
    xgb_model.load_model(f'/kaggle/input/models/t8101349/prime-model/pytorch/default/1/xgb_fold_{fold}.json')
    preds_xgb += xgb_model.predict(dtest_xgb) / 5.0

print("✅ 樹模型 5-Fold 推理完成！")


In [ ]:
# --- B. CNN 推理 ---
print("🔮 正在進行 CNN 推理 (Batch Mode)...")

# 建立 DataLoader (你的寫法完全正確)
cnn_dataset = TensorDataset(
    torch.from_numpy(smiles_seqs).long(),
    torch.from_numpy(protein_ids).long(),
    torch.from_numpy(protein_numerics).float()
)
# 推論時不需要計算梯度，可以把 batch_size 開大一點加速 (例如 2048)
cnn_loader = DataLoader(cnn_dataset, batch_size=2048, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 準備空陣列來裝 5 折平均結果
preds_cnn = np.zeros(len(cnn_dataset))

# 建立一個空的模型架構 (記得傳入正確的 vocab_size)
cnn_model = BelkaFusionModel(vocab_size=vocab_size).to(device)

for fold in range(1, 6):
    print(f"📥 載入 CNN Fold {fold}...")
    
    # 1. 讀取並灌入權重 (map_location 確保即使在 CPU 環境也不會報錯)
    cnn_model.load_state_dict(torch.load(f'/kaggle/input/models/t8101349/prime-model/pytorch/default/1/cnn_fold_{fold}.pt', map_location=device))
    cnn_model.eval() # 切換到推論模式！這對 Batch Norm 非常重要
    
    fold_preds = []
    
    # 2. 分批預測
    with torch.no_grad(): # 關閉梯度計算，省下大量 VRAM
        for seqs, p_ids, p_nums in cnn_loader:
            seqs, p_ids, p_nums = seqs.to(device), p_ids.to(device), p_nums.to(device)
            
            # 取得機率值輸出
            out = cnn_model(seqs, p_ids, p_nums)
            
            # 將 tensor 轉回 numpy array 並收集起來
            fold_preds.extend(out.cpu().numpy().flatten())
            
    # 3. 將這個 Fold 的預測結果加入總和並平均
    preds_cnn += np.array(fold_preds) / 5.0

print("✅ CNN 5-Fold 推理完成！")


In [32]:
# 方法二reg_meta_final_preds
# 將預測結果對齊並合併為矩陣 (Shape: [1600000, 3])
df_test = pd.read_csv("/kaggle/input/leash-BELKA/test.csv", usecols=['protein_name'])
protein_map = {'sEH': 0, 'BRD4': 1, 'HSA': 2}
test_protein_ids = df_test['protein_name'].map(protein_map).values.astype(np.int8)

# ==========================================
# 1. 建立與訓練時【一模一樣】的 DataFrame
# ==========================================
# 🚨 直接使用字典建立 DataFrame，不要用 np.column_stack
X_meta_test = pd.DataFrame({
    'pred_lgb': preds_lgb,
    'pred_xgb': preds_xgb,
    'pred_cnn': preds_cnn,
    'protein_id': test_protein_ids
})

# 轉換為 category 型態，完美對齊訓練時的格式
X_meta_test['protein_id'] = X_meta_test['protein_id'].astype('category')

print(f"📊 Meta-feature shape: {X_meta_test.shape}")

# ==========================================
# 2. 執行 10-Fold 模型平均推理 (Ensemble)
# ==========================================
print("🚀 啟動 10-Fold Meta-LGBM 推理...")

# 準備空陣列裝預測結果
reg_meta_final_preds = np.zeros(len(X_meta_test))

# 迴圈讀取 10 個 txt 模型
for fold in range(1, 11):
    print(f"📥 載入 Meta-LGBM Fold {fold} 並預測...")
    meta_model = lgb.Booster(model_file=f'meta_lgbm_fold_{fold}.txt')
    
    # 累加預測機率，並除以 10 作平均
    reg_meta_final_preds += meta_model.predict(X_meta_test) / 10.0

# 確保機率值在合理範圍 (0~1)
reg_meta_final_preds = np.clip(reg_meta_final_preds, 0, 1)

📊 Meta-feature shape: (1674896, 4)
🚀 啟動 10-Fold Meta-LGBM 推理...
📥 載入 Meta-LGBM Fold 1 並預測...
📥 載入 Meta-LGBM Fold 2 並預測...
📥 載入 Meta-LGBM Fold 3 並預測...
📥 載入 Meta-LGBM Fold 4 並預測...
📥 載入 Meta-LGBM Fold 5 並預測...
📥 載入 Meta-LGBM Fold 6 並預測...
📥 載入 Meta-LGBM Fold 7 並預測...
📥 載入 Meta-LGBM Fold 8 並預測...
📥 載入 Meta-LGBM Fold 9 並預測...
📥 載入 Meta-LGBM Fold 10 並預測...


In [33]:
import gc

try:
    del reg_meta_test
except:
    pass
gc.collect()

print("📝 正在封裝 Submission 檔案...")

# 建立最終 DataFrame
submissionreg = pd.DataFrame({
    'id': test_ids,
    'binds': reg_meta_final_preds
})

# 存檔
submissionreg.to_csv('submissionreg.csv', index=False)

print("---")
print("提交檔 'submissionreg.csv' 已準備就緒。")
# 最後檢查行數是否正確
sample_sub = pd.read_csv("/kaggle/input/leash-BELKA/sample_submission.csv", usecols=['id'])
if len(submissionreg) == len(sample_sub):
    print(f"✅ 行數檢查通過: {len(submission)} 筆數據")
else:
    print(f"❌ 警告：行數不符！提交檔: {len(submission)}, 範本: {len(sample_sub)}")

📝 正在封裝 Submission 檔案...
---
提交檔 'submissionreg.csv' 已準備就緒。
✅ 行數檢查通過: 1674896 筆數據


In [41]:
import zipfile
import os

# 把所有的 submission 相關檔案打包
files_to_download = ['submission.csv', 'submission_lgb.csv', 'submissionreg.csv']

with zipfile.ZipFile('all_submissions.zip', 'w') as zipf:
    for file in files_to_download:
        if os.path.exists(file):
            zipf.write(file)
            print(f"✅ 已加入打包: {file}")

from IPython.display import FileLink
display(FileLink('all_submissions.zip'))

✅ 已加入打包: submission.csv
✅ 已加入打包: submission_lgb.csv
✅ 已加入打包: submissionreg.csv


/kaggle/working/all_submissions.zip